# 08 — Experimentos

Este notebook compara variantes ligeras del modelo VLA. Todos los experimentos usan exactamente las mismas particiones de **train** y **validation**; el conjunto de **test** queda reservado para el notebook 09.

## 1. Configuración e imports

Se reutilizan los embeddings cacheados de CLIP del notebook 04. CLIP no se vuelve a entrenar: solo se entrenan el Transformer de fusión y el decodificador.

In [ ]:
from pathlib import Path
import json
import random
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT_DIR = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / 'src').exists() else NOTEBOOK_DIR
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.evaluation import benchmark_inference, evaluate_actions
from src.project_config import CACHE_DIR, PROJECT_DIR, experiment_dirs
from src.train import load_checkpoint, train_model
from src.vla_model import VLA

ROOT_DIR = PROJECT_DIR
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 16
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS_MAX = 50
PATIENCE = 7
# None usa todas las muestras. Puede cambiarse por un entero para una prueba rápida.
MAX_TRAIN_SAMPLES = None

def fijar_semilla(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

fijar_semilla()
RUTAS = experiment_dirs('experimentos_vla')
print(f'Dispositivo: {DEVICE} | semilla: {SEED}')
print(f'Resultados: {RUTAS["results"]}')

## 2. Carga de datos y preparación de lotes

Se cargan únicamente `train` y `validation`. Las acciones tienen el orden `[terminate, x, y, z, rx, ry, rz, gripper]`.

In [ ]:
EMBEDDING_DIM = 512
OUTPUT_DIM = 8

def cargar_particion(nombre):
    ruta = CACHE_DIR / f'{nombre}.npz'
    if not ruta.exists():
        raise FileNotFoundError(f'No existe {ruta}. Ejecuta antes 04_clip_embeddings.ipynb.')
    with np.load(ruta) as datos:
        requeridas = {'imagenes_static', 'imagenes_gripper', 'textos', 'acciones'}
        if not requeridas.issubset(datos.files):
            raise KeyError(f'{nombre}: se esperaban las claves {requeridas}')
        arrays = tuple(np.asarray(datos[clave], dtype=np.float32) for clave in
                       ('imagenes_static', 'imagenes_gripper', 'textos', 'acciones'))
    static, gripper, texto, acciones = arrays
    if not (len(static) == len(gripper) == len(texto) == len(acciones)):
        raise ValueError(f'{nombre}: longitudes inconsistentes')
    if static.shape[1:] != (EMBEDDING_DIM,) or gripper.shape[1:] != (EMBEDDING_DIM,) or texto.shape[1:] != (EMBEDDING_DIM,):
        raise ValueError(f'{nombre}: embeddings con dimensión inesperada')
    if acciones.shape[1:] != (OUTPUT_DIM,) or not all(np.isfinite(x).all() for x in arrays):
        raise ValueError(f'{nombre}: acciones o embeddings no válidos')
    return arrays

particiones = {nombre: cargar_particion(nombre) for nombre in ('train', 'validation')}
if MAX_TRAIN_SAMPLES is not None:
    particiones['train'] = tuple(array[:MAX_TRAIN_SAMPLES] for array in particiones['train'])

for nombre, (static, gripper, texto, acciones) in particiones.items():
    print(f'{nombre:10s}: static={static.shape}, gripper={gripper.shape}, texto={texto.shape}, acciones={acciones.shape}')

with (ROOT_DIR / 'data' / 'parametros_normalizacion.json').open(encoding='utf-8') as archivo:
    normalizacion = json.load(archivo)
MINIMO = np.asarray(normalizacion['minimo'], dtype=np.float32)
ESCALA = np.asarray(normalizacion['escala'], dtype=np.float32)

## 3. Funciones reutilizables

La misma función entrena, recupera el mejor checkpoint y calcula las métricas de validación de cada variante. Esto evita repetir código y garantiza que el procedimiento sea igual en todos los casos.

In [ ]:
def aplicar_modalidades(arrays, modalidad):
    """Anula modalidades para conservar la misma arquitectura de tres tokens."""
    static, gripper, texto, acciones = arrays
    if modalidad == 'imagen':
        texto = np.zeros_like(texto)
    elif modalidad == 'texto':
        static = np.zeros_like(static)
        gripper = np.zeros_like(gripper)
    elif modalidad != 'imagen_texto':
        raise ValueError(f'Modalidad no reconocida: {modalidad}')
    return static, gripper, texto, acciones

def crear_loaders(train_arrays, validation_arrays):
    train_dataset = TensorDataset(*(torch.from_numpy(array) for array in train_arrays))
    validation_dataset = TensorDataset(*(torch.from_numpy(array) for array in validation_arrays))
    # Se reinicia el generador para que el orden de train sea comparable.
    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator, num_workers=0)
    validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return train_loader, validation_loader

def crear_modelo(config):
    return VLA(
        clip_encoder=None, embedding_dim=EMBEDDING_DIM, fusion_dim=config['fusion_dim'],
        decoder_hidden_dim=128, num_layers=config['num_layers'], num_heads=4,
        feedforward_dim=config['fusion_dim'] * 2, dropout=0.1,
    ).to(DEVICE)

def predecir(modelo, loader):
    predicciones, objetivos = [], []
    modelo.eval()
    with torch.no_grad():
        for static, gripper, texto, accion in loader:
            salida = modelo(static_embeddings=static.to(DEVICE), gripper_embeddings=gripper.to(DEVICE), text_embeddings=texto.to(DEVICE))
            predicciones.append(salida.cpu())
            objetivos.append(accion)
    return torch.cat(predicciones).numpy(), torch.cat(objetivos).numpy()

def ejecutar_experimento(config):
    fijar_semilla()
    nombre = config['name']
    rutas = experiment_dirs(f'experimentos_vla/{nombre}')
    train_arrays = aplicar_modalidades(particiones['train'], config['modalidad'])
    validation_arrays = aplicar_modalidades(particiones['validation'], config['modalidad'])
    train_loader, validation_loader = crear_loaders(train_arrays, validation_arrays)
    modelo = crear_modelo(config)
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    checkpoint_path = rutas['checkpoints'] / 'best.pt'
    configuracion = {**config, 'seed': SEED, 'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
                     'weight_decay': WEIGHT_DECAY, 'epochs_max': EPOCHS_MAX, 'patience': PATIENCE}
    resumen = train_model(modelo, train_loader, validation_loader, optimizador, DEVICE,
                           epochs=EPOCHS_MAX, patience=PATIENCE, checkpoint_path=checkpoint_path, config=configuracion)
    recuperado = crear_modelo(config)
    checkpoint = load_checkpoint(checkpoint_path, recuperado, map_location=DEVICE)
    y_pred, y_true = predecir(recuperado, validation_loader)
    metricas = evaluate_actions(y_pred, y_true, MINIMO, ESCALA)
    total = sum(p.numel() for p in recuperado.parameters())
    entrenables = sum(p.numel() for p in recuperado.parameters() if p.requires_grad)
    n_tiempo = min(32, len(validation_arrays[0]))
    entrada_tiempo = [torch.from_numpy(array[:n_tiempo]).to(DEVICE) for array in validation_arrays[:3]]
    inferencia = benchmark_inference(lambda: recuperado(static_embeddings=entrada_tiempo[0], gripper_embeddings=entrada_tiempo[1], text_embeddings=entrada_tiempo[2]), n_tiempo)
    resultado = {**configuracion, 'best_epoch': resumen['best_epoch'],
                 'best_validation_loss': resumen['best_validation_loss'],
                 'epochs_executed': resumen['epochs_executed'],
                 'total_training_time_seconds': resumen['total_training_time_seconds'],
                 'parameters_total': total, 'parameters_trainable': entrenables,
                 'validation_metrics': metricas, 'cached_inference': inferencia,
                 'checkpoint': str(checkpoint_path), 'history': resumen['history'], 'test_used': False}
    with (rutas['results'] / 'resultado.json').open('w', encoding='utf-8') as archivo:
        json.dump(resultado, archivo, indent=2, ensure_ascii=False)
    print(f'{nombre}: validation loss={resultado["best_validation_loss"]:.6f} | época={resultado["best_epoch"]}')
    return resultado

## 4. Configuraciones a comparar

Se entrena una referencia VLA estándar y cinco variantes. La referencia (2 capas y dimensión 256) se reutiliza en la comparación de capas, dimensión y modalidades, por lo que no se entrena dos veces.

In [ ]:
EXPERIMENTOS = [
    {'name': 'vla_referencia', 'grupo': 'referencia', 'descripcion': 'VLA estándar', 'num_layers': 2, 'fusion_dim': 256, 'modalidad': 'imagen_texto'},
    {'name': 'capas_1', 'grupo': 'capas', 'descripcion': '1 capa', 'num_layers': 1, 'fusion_dim': 256, 'modalidad': 'imagen_texto'},
    {'name': 'capas_3', 'grupo': 'capas', 'descripcion': '3 capas', 'num_layers': 3, 'fusion_dim': 256, 'modalidad': 'imagen_texto'},
    {'name': 'dimension_128', 'grupo': 'dimension', 'descripcion': 'd_model = 128', 'num_layers': 2, 'fusion_dim': 128, 'modalidad': 'imagen_texto'},
    {'name': 'solo_imagen', 'grupo': 'modalidades', 'descripcion': 'Solo imagen', 'num_layers': 2, 'fusion_dim': 256, 'modalidad': 'imagen'},
    {'name': 'solo_texto', 'grupo': 'modalidades', 'descripcion': 'Solo texto', 'num_layers': 2, 'fusion_dim': 256, 'modalidad': 'texto'},
]
pd.DataFrame(EXPERIMENTOS)[['name', 'grupo', 'descripcion', 'num_layers', 'fusion_dim', 'modalidad']]

## 5. Ejecución de los experimentos

Cada configuración empieza con pesos iniciales y orden de entrenamiento reproducibles. En CPU, el conjunto completo puede requerir bastante tiempo; `MAX_TRAIN_SAMPLES` permite una ejecución de comprobación más corta sin modificar el flujo.

In [ ]:
resultados = [ejecutar_experimento(config) for config in EXPERIMENTOS]

## 6. Comparación con el baseline MLP

El baseline se entrena en el notebook 05. Si su archivo de resultados de validación está disponible, se añade como referencia; si no, el notebook sigue siendo ejecutable y muestra una advertencia.

In [ ]:
def cargar_baseline():
    ruta = ROOT_DIR / 'results' / 'baseline_mlp_binary_phase1' / 'baseline_mlp_binary_validation.json'
    if not ruta.exists():
        print('Baseline no disponible: ejecuta 05_baseline_mlp.ipynb para añadirlo a la comparación.')
        return None
    with ruta.open(encoding='utf-8') as archivo:
        baseline = json.load(archivo)
    return {
        'name': 'baseline_mlp', 'grupo': 'baseline', 'descripcion': 'MLP del notebook 05',
        'best_validation_loss': baseline['best_validation_loss'],
        'parameters_trainable': baseline['computational_efficiency']['trainable_parameters'],
        'validation_metrics': baseline['validation_action_metrics'],
    }

baseline = cargar_baseline()
if baseline is not None:
    resultados.append(baseline)

## 7. Resumen y guardado

La tabla resume las métricas principales de validación. El tiempo de inferencia corresponde solo al VLA sobre embeddings cacheados, porque CLIP está congelado e idéntico en todas las variantes; la medición completa del sistema se realizará en el notebook 09.

In [ ]:
def fila_resumen(resultado):
    metricas = resultado['validation_metrics']
    fila = {
        'experimento': resultado['name'], 'grupo': resultado['grupo'], 'descripcion': resultado['descripcion'],
        'validation_loss': resultado['best_validation_loss'],
        'mae': metricas['continuous_normalized']['mae'],
        'rmse': metricas['continuous_normalized']['rmse'],
        'coseno_xyz': metricas['xyz_denormalized_cosine_similarity'],
        'terminate_f1': metricas['terminate']['f1'],
        'terminate_accuracy': metricas['terminate']['accuracy'],
        'parametros_entrenables': resultado['parameters_trainable'],
    }
    if 'cached_inference' in resultado:
        fila['inferencia_ms_muestra'] = resultado['cached_inference']['mean_ms_per_sample']
        fila['tiempo_entrenamiento_s'] = resultado['total_training_time_seconds']
    return fila

tabla_resultados = pd.DataFrame([fila_resumen(resultado) for resultado in resultados])
tabla_resultados = tabla_resultados.sort_values(['grupo', 'experimento']).reset_index(drop=True)
tabla_path = RUTAS['results'] / 'resumen_experimentos.csv'
json_path = RUTAS['results'] / 'resumen_experimentos.json'
tabla_resultados.to_csv(tabla_path, index=False)
with json_path.open('w', encoding='utf-8') as archivo:
    json.dump(resultados, archivo, indent=2, ensure_ascii=False)

display(tabla_resultados.round(4))
print(f'Resumen guardado en {tabla_path}')
print('El conjunto de test no se ha usado. El notebook 09 evaluará el modelo seleccionado.')

## Resumen

El notebook ha comparado la profundidad y dimensión del Transformer, así como la aportación de imagen y texto. Los resultados quedan disponibles para seleccionar la configuración final antes de evaluarla sobre test en el notebook 09 y generar las gráficas definitivas en el notebook 10.